# 4. Treinamento — Segmentação de Pessoas e EPIs

**Objetivo**

Nesta etapa será treinado um modelo de segmentação de instâncias
utilizando YOLO Ultralytics.

Diferentemente da etapa de detecção, que utiliza bounding boxes,
o modelo de segmentação utilizará as máscaras poligonais originais
das imagens.

O objetivo é identificar os objetos e delimitar sua área de forma
mais precisa, permitindo comparar os resultados de detecção e
segmentação no mesmo domínio.

O treinamento será realizado utilizando fine-tuning a partir de
pesos pré-treinados.

**4.1 Preparação do ambiente**

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU não disponível.")

In [ ]:
!pip install -q ultralytics

In [ ]:
import ultralytics

print("Ultralytics:", ultralytics.__version__)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
import zipfile

# Procurar o ZIP enviado
arquivos_zip = [
    f for f in os.listdir("/content")
    if f.lower().endswith(".zip")
]

print("Arquivos ZIP encontrados:")
for arquivo in arquivos_zip:
    print("-", arquivo)

# Caminho do ZIP
zip_path = os.path.join("/content", arquivos_zip[0])

# Pasta de extração
extract_path = "/content/PPE_dataset"

# Extrair
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("\nExtração concluída! ✅")
print("Pasta:", extract_path)

In [ ]:
import os
import shutil

original_path = "/content/PPE_dataset"
clean_path = "/content/PPE_dataset_clean"

# Remover cópia anterior, se existir
if os.path.exists(clean_path):
    shutil.rmtree(clean_path)

# Criar cópia
shutil.copytree(original_path, clean_path)

print("Cópia limpa criada em:")
print(clean_path)

# Arquivos identificados anteriormente como problemáticos
arquivos_problematicos = {
    "train": [
        "frameHRGateDay-6-mp422000_jpg.rf.3d2f293e76e6684116e1f37927287c4f.txt",
        "frameHRGateDay-6-mp422000_jpg.rf.8b2b90e67dfdb25c41b31618b7f13a15.txt",
        "frameHRGateDay-6-mp422000_jpg.rf.8f6838da4874ddb2b77afe3157fd8f47.txt"
    ],
    "valid": [
        "frameHRGateDay-6-mp421000_jpg.rf.0a848b789a602dfcb85188607dc2d8ea.txt"
    ],
    "test": []
}

# Remover os 4 casos problemáticos
for split, arquivos in arquivos_problematicos.items():

    labels_dir = os.path.join(clean_path, split, "labels")
    images_dir = os.path.join(clean_path, split, "images")

    for label_file in arquivos:

        label_path = os.path.join(labels_dir, label_file)

        if os.path.exists(label_path):
            os.remove(label_path)

        # Nome da imagem correspondente
        image_base = os.path.splitext(label_file)[0]

        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:

            image_path = os.path.join(
                images_dir,
                image_base + ext
            )

            if os.path.exists(image_path):
                os.remove(image_path)

print("Casos problemáticos removidos. ✅")

In [ ]:
#validar as anotações de segmentação
import os

image_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

erros = []

for split in ["train", "valid", "test"]:

    images_dir = os.path.join(
        clean_path, split, "images"
    )

    labels_dir = os.path.join(
        clean_path, split, "labels"
    )

    imagens = [
        f for f in os.listdir(images_dir)
        if f.lower().endswith(image_extensions)
    ]

    labels = [
        f for f in os.listdir(labels_dir)
        if f.endswith(".txt")
    ]

    print(f"\n{split}:")
    print("  Imagens:", len(imagens))
    print("  Labels:", len(labels))

    # Verificar imagem sem label
    for imagem in imagens:

        nome = os.path.splitext(imagem)[0]
        label_path = os.path.join(
            labels_dir,
            nome + ".txt"
        )

        if not os.path.exists(label_path):
            erros.append(
                f"{split}: imagem sem label -> {imagem}"
            )

    # Verificar formato dos labels
    for label_file in labels:

        caminho = os.path.join(
            labels_dir,
            label_file
        )

        with open(caminho, "r") as f:
            linhas = f.readlines()

        if len(linhas) == 0:
            erros.append(
                f"{split}: label vazio -> {label_file}"
            )
            continue

        for numero, linha in enumerate(linhas, start=1):

            valores = linha.strip().split()

            # Segmentação precisa ter:
            # classe + pares X,Y
            if len(valores) < 7:
                erros.append(
                    f"{split}: {label_file}, "
                    f"linha {numero}: poucos valores"
                )
                continue

            # Depois da classe devem existir pares
            # de coordenadas
            quantidade_coordenadas = len(valores) - 1

            if quantidade_coordenadas % 2 != 0:
                erros.append(
                    f"{split}: {label_file}, "
                    f"linha {numero}: "
                    f"coordenadas não formam pares"
                )
                continue

            try:

                classe = int(valores[0])

                if classe not in range(5):
                    erros.append(
                        f"{split}: {label_file}, "
                        f"classe inválida: {classe}"
                    )

                coordenadas = [
                    float(v)
                    for v in valores[1:]
                ]

                for valor in coordenadas:

                    if valor < 0 or valor > 1:
                        erros.append(
                            f"{split}: {label_file}, "
                            f"coordenada fora de 0-1"
                        )

            except ValueError:

                erros.append(
                    f"{split}: {label_file}, "
                    f"valor inválido"
                )

print("\n" + "=" * 50)
print("VALIDAÇÃO DO DATASET DE SEGMENTAÇÃO")
print("=" * 50)

print("Quantidade de erros:", len(erros))

if erros:
    print("\nPrimeiros erros:")
    for erro in erros[:20]:
        print("-", erro)
else:
    print("Dataset de segmentação validado com sucesso! ✅")

**4.2 Criar data.yaml de segmentação**

In [ ]:
yaml_content = """train: /content/PPE_dataset_clean/train/images
val: /content/PPE_dataset_clean/valid/images
test: /content/PPE_dataset_clean/test/images

nc: 5

names:
  0: person
  1: with-helmet
  2: with-suit
  3: without-helmet
  4: without-suit
"""

yaml_path = "/content/PPE_dataset_clean/data_segmentation.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("data_segmentation.yaml criado com sucesso! ✅")
print("\nConteúdo:")
print(yaml_content)

**4.2 Carregar o YOLO11n-seg**

In [ ]:
from ultralytics import YOLO

# Carregar modelo de segmentação pré-treinado
seg_model = YOLO("yolo11n-seg.pt")

print("Modelo YOLO11n-seg carregado com sucesso! ✅")

**4.3 Máscara do dataset**

In [ ]:
import os
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Classes
class_names = {
    0: "person",
    1: "with-helmet",
    2: "with-suit",
    3: "without-helmet",
    4: "without-suit"
}

images_dir = "/content/PPE_dataset_clean/train/images"
labels_dir = "/content/PPE_dataset_clean/train/labels"

# Escolher uma imagem aleatória
imagens = [
    f for f in os.listdir(images_dir)
    if f.lower().endswith(image_extensions)
]

nome_imagem = random.choice(imagens)

imagem_path = os.path.join(images_dir, nome_imagem)
label_path = os.path.join(
    labels_dir,
    os.path.splitext(nome_imagem)[0] + ".txt"
)

# Ler imagem
imagem = cv2.imread(imagem_path)
imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)

altura, largura = imagem.shape[:2]

# Contador de máscaras
quantidade_mascaras = 0

# Ler labels
with open(label_path, "r") as f:
    linhas = f.readlines()

# Desenhar os polígonos
for linha in linhas:

    valores = linha.strip().split()

    classe = int(valores[0])

    coordenadas = [
        float(v) for v in valores[1:]
    ]

    pontos = []

    for i in range(0, len(coordenadas), 2):

        x = int(coordenadas[i] * largura)
        y = int(coordenadas[i + 1] * altura)

        pontos.append([x, y])

    pontos = np.array(
        pontos,
        dtype=np.int32
    ).reshape((-1, 1, 2))

    # Desenhar contorno
    cv2.polylines(
        imagem,
        [pontos],
        True,
        (255, 0, 0),
        2
    )

    # Nome da classe
    x_texto = pontos[0][0][0]
    y_texto = max(
        pontos[0][0][1] - 5,
        15
    )

    cv2.putText(
        imagem,
        class_names[classe],
        (x_texto, y_texto),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 0, 0),
        2
    )

    quantidade_mascaras += 1

# IMPORTANTE:
# Exibir somente depois de desenhar as máscaras
plt.figure(figsize=(10, 10))
plt.imshow(imagem)
plt.axis("off")
plt.title(
    f"{nome_imagem} — {quantidade_mascaras} máscaras"
)
plt.show()

print("Imagem:", nome_imagem)
print("Máscaras encontradas:", quantidade_mascaras)

**4.4 Treinamento do YOLO11n-seg**

In [ ]:
seg_results = seg_model.train(
    data="/content/PPE_dataset_clean/data_segmentation.yaml",

    epochs=30,
    imgsz=640,
    batch=16,

    device=0,
    seed=42,

    project="/content/results/segmentation",
    name="yolo11n_seg_ppe",

    patience=10,

    plots=True,
    verbose=True
)

**4.5 Teste final da segmentação**

In [ ]:
from ultralytics import YOLO

# Carregar o melhor modelo de segmentação
best_seg_model = YOLO(
    "/content/results/segmentation/yolo11n_seg_ppe/weights/best.pt"
)

# Avaliar no conjunto de teste
seg_test_metrics = best_seg_model.val(
    data="/content/PPE_dataset_clean/data_segmentation.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/content/results/segmentation",
    name="test"
)

print("\nAvaliação da segmentação no conjunto de TESTE concluída! ✅")

**4.6 Análise qualitativa da segmentação**

In [ ]:
import os

results_path = "/content/results/segmentation/yolo11n_seg_ppe"

print("Arquivos gerados pelo treinamento:\n")

for root, dirs, files in os.walk(results_path):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
#val batch0
from IPython.display import display
from PIL import Image

base_path = "/content/results/segmentation/yolo11n_seg_ppe"

print("MÁSCARAS REAIS — BATCH 0")
display(
    Image.open(
        f"{base_path}/val_batch0_labels.jpg"
    )
)

print("MÁSCARAS PREVISTAS — BATCH 0")
display(
    Image.open(
        f"{base_path}/val_batch0_pred.jpg"
    )
)

In [ ]:
#val batch1_2
from IPython.display import display
from PIL import Image

base_path = "/content/results/segmentation/yolo11n_seg_ppe"

print("MÁSCARAS REAIS — BATCH 1")
display(
    Image.open(
        f"{base_path}/val_batch1_labels.jpg"
    )
)

print("MÁSCARAS PREVISTAS — BATCH 1")
display(
    Image.open(
        f"{base_path}/val_batch1_pred.jpg"
    )
)

print("\n" + "=" * 60 + "\n")

print("MÁSCARAS REAIS — BATCH 2")
display(
    Image.open(
        f"{base_path}/val_batch2_labels.jpg"
    )
)

print("MÁSCARAS PREVISTAS — BATCH 2")
display(
    Image.open(
        f"{base_path}/val_batch2_pred.jpg"
    )
)

**4.7 Histórico de segmentação**

In [ ]:
import pandas as pd

results_csv_seg = (
    "/content/results/segmentation/"
    "yolo11n_seg_ppe/results.csv"
)

df_seg = pd.read_csv(results_csv_seg)

# Remover espaços dos nomes das colunas
df_seg.columns = df_seg.columns.str.strip()

print("Colunas disponíveis:")
for coluna in df_seg.columns:
    print("-", coluna)

print("\nÚltimas 5 épocas:")
display(df_seg.tail())

In [ ]:
# Identificar as melhores épocas das métricas de máscara

melhor_precision_m = df_seg.loc[
    df_seg["metrics/precision(M)"].idxmax()
]

melhor_recall_m = df_seg.loc[
    df_seg["metrics/recall(M)"].idxmax()
]

melhor_map50_m = df_seg.loc[
    df_seg["metrics/mAP50(M)"].idxmax()
]

melhor_map5095_m = df_seg.loc[
    df_seg["metrics/mAP50-95(M)"].idxmax()
]

print("MELHORES RESULTADOS DE SEGMENTAÇÃO DURANTE O TREINAMENTO")
print("=" * 65)

print(
    f"Melhor Precision (Mask): "
    f"época {int(melhor_precision_m['epoch']) + 1} "
    f"→ {melhor_precision_m['metrics/precision(M)']:.4f}"
)

print(
    f"Melhor Recall (Mask): "
    f"época {int(melhor_recall_m['epoch']) + 1} "
    f"→ {melhor_recall_m['metrics/recall(M)']:.4f}"
)

print(
    f"Melhor mAP@0.5 (Mask): "
    f"época {int(melhor_map50_m['epoch']) + 1} "
    f"→ {melhor_map50_m['metrics/mAP50(M)']:.4f}"
)

print(
    f"Melhor mAP@0.5:0.95 (Mask): "
    f"época {int(melhor_map5095_m['epoch']) + 1} "
    f"→ {melhor_map5095_m['metrics/mAP50-95(M)']:.4f}"
)